In [0]:
from pyspark.sql.functions import *
rma_df = spark.table("workspace.default.rma_county_yields_report_399")
noaa_df = spark.table("workspace.default.noaa_station_month_metrics")

rma_df = rma_df.withColumn("State Abbreviation", trim(col("State Abbreviation")))
noaa_df = noaa_df.withColumn("state", trim(col("state")))

merged_df = rma_df.join(
    noaa_df,
    (rma_df["State Abbreviation"] == noaa_df["state"]) &
    (rma_df["Yield Year"] == noaa_df["year"]),
    "inner"
)

display(merged_df)

In [0]:
# Filter months 4-9
growing_season_df = merged_df.filter(col("month").between(4, 9))

# Assign growth stage category
growing_season_df = growing_season_df.withColumn(
    "Growth_Stage",
    when(col("month").isin([4, 5]), "Emergence")
    .when(col("month").isin([5, 6]), "Vegetative Growth")
    .when(col("month") == 7, "Pollination")
    .when(col("month").isin([8, 9]), "Grain Fill")
)

display(growing_season_df)

In [0]:
selected_df = growing_season_df.select(
    col("State Name"),
    col("Yield Year"),
    col("Yield Amount"),
    col("monthly_rainfall_mm"),
    col("Growth_Stage")
)

display(selected_df)

In [0]:
deduped_df = selected_df.dropDuplicates(["State Name", "Yield Year", "Growth_Stage"])

ordered_df = deduped_df.orderBy("State Name")


display(ordered_df)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import avg

# Group by State Name and Growth_Stage, compute average Yield Amount and monthly_rainfall_mm
avg_df = selected_df.groupBy("State Name", "Growth_Stage").agg(
    avg("Yield Amount").alias("Avg_Yield_Amount"),
    avg("monthly_rainfall_mm").alias("Avg_Monthly_Rainfall_mm")
).orderBy("State Name")

display(avg_df)

In [0]:
from pyspark.sql.functions import corr

# Compute correlation between Yield Amount and monthly_rainfall_mm for each State Name and Growth_Stage
corr_df = selected_df.groupBy("State Name", "Growth_Stage").agg(
    corr("Yield Amount", "monthly_rainfall_mm").alias("Yield_Rainfall_Correlation")
).orderBy("State Name")

display(corr_df)

Databricks visualization. Run in Databricks to view.